#### MongoBase starting guide

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import sys
import time
import threading
import multiprocessing
import datetime as dt
sys.path.append('../')
from models.mongobase import MongoBase, db_context
from bson import ObjectId

#### ObjectId

First, let's talk about ObjectId.

In [2]:
x = ObjectId()
time.sleep(1)
y = ObjectId()
time.sleep(1)
z = ObjectId()

In [3]:
x

ObjectId('5c8f7ca4144682821ddf40cd')

In [4]:
str(x)

'5c8f7ca4144682821ddf40cd'

In [5]:
x.generation_time

datetime.datetime(2019, 3, 18, 11, 10, 28, tzinfo=<bson.tz_util.FixedOffset object at 0x7fbc028381d0>)

In [6]:
y.generation_time

datetime.datetime(2019, 3, 18, 11, 10, 29, tzinfo=<bson.tz_util.FixedOffset object at 0x7fbc028381d0>)

In [7]:
x < y and y < z

True

Actually, ObjectId is usuful. It is unique, sortable and memory efficient.

http://api.mongodb.com/python/current/api/bson/objectid.html

>An ObjectId is a 12-byte unique identifier consisting of:

>a 4-byte value representing the seconds since the Unix epoch,  
>a 3-byte machine identifier,  
>a 2-byte process id, and  
>a 3-byte counter, starting with a random value.  

And also ObjectId is fast for inserting or indexing. The index size is small.

https://github.com/Restuta/mongo.Guid-vs-ObjectId-performance

#### Define a database model

So now, we create a simple test collection with MongoBase.

In [8]:
class Bird(MongoBase):
    __collection__ = 'birds'
    __structure__ = {
        '_id': ObjectId,
        'name': str,
        'age': int,
        'is_able_to_fly': bool,
        'created': dt.datetime,
        'updated': dt.datetime
    }
    __required_fields__ = ['_id', 'name']
    __default_values__ = {
        '_id': ObjectId(),
        'is_able_to_fly': False
    }
    __validators__ = {}
    __indexed_keys__ = {}

The `__structure__` part represents the definition of the model.


【NOTE】
If you need a substructure, you may can as below. Hoever, in most cases, other independent models should be defined. 

```py

class Queen(MongoBase):
    __collection__ = 'queens'
    child = {
        'name': str,
        'age': int
    }
    __structure__ = {
        '_id': ObjectId,
        'name': str,
        'inheritor': dict,  # child
        'children': list,  # [child]
    }
```



#### Basic instractions. (insert, update, find, remove)

Let's try basic instractions like inserts, updates, find and remove. 

Firstly, let's begin with creating an instance to be stored.

In [9]:
chicken = Bird({'_id': ObjectId(), 'name': 'chicken', 'age': 3})

In [10]:
chicken

{'_id': ObjectId('5c8f7ca6144682821ddf40d1'),
 'name': 'chicken',
 'age': 3,
 'is_able_to_fly': False,
 'created': None,
 'updated': None}

In [11]:
chicken._id.generation_time

datetime.datetime(2019, 3, 18, 11, 10, 30, tzinfo=<bson.tz_util.FixedOffset object at 0x7fbc028381d0>)

Good chicken. Let's save while it is fresh.

In [12]:
chicken.save()

{'_id': ObjectId('5c8f7ca6144682821ddf40d1'),
 'name': 'chicken',
 'age': 3,
 'is_able_to_fly': False,
 'created': datetime.datetime(2019, 3, 18, 11, 10, 30, 995450, tzinfo=datetime.timezone.utc),
 'updated': datetime.datetime(2019, 3, 18, 11, 10, 30, 995450, tzinfo=datetime.timezone.utc)}

Chickens are considered to be unable to fly by default. We can let it be enable by updating.

In [13]:
chicken.is_able_to_fly

False

In [14]:
chicken.is_able_to_fly = True
chicken.update()

{'_id': ObjectId('5c8f7ca6144682821ddf40d1'),
 'name': 'chicken',
 'age': 3,
 'is_able_to_fly': True,
 'created': datetime.datetime(2019, 3, 18, 11, 10, 30, 995450, tzinfo=datetime.timezone.utc),
 'updated': datetime.datetime(2019, 3, 18, 11, 10, 31, 86542, tzinfo=datetime.timezone.utc)}

In [15]:
Bird.findOne({'created': chicken.created})

{'_id': ObjectId('5c8f7ca6144682821ddf40d1'),
 'name': 'chicken',
 'age': 3,
 'is_able_to_fly': True,
 'created': datetime.datetime(2019, 3, 18, 11, 10, 30, 995000),
 'updated': datetime.datetime(2019, 3, 18, 11, 10, 31, 86000)}

You would be able to see `'is_able_to_fly': True`.  

Chickens grow up in several ways.

In [16]:
chicken.age = 5
chicken = chicken.update()
assert chicken.age == 5, 'something wrong on update()'
chicken = Bird.findAndUpdateById(chicken._id, {'age': 6})
assert chicken.age == 6, 'something wrong on findAndUpdateById()'

Next let's try find methods.

In [17]:
mother_chicken = Bird({'_id': ObjectId(), 'name': 'mother chicken', 'age': 63})
mother_chicken.save()

{'_id': ObjectId('5c8f7ca7144682821ddf40d2'),
 'name': 'mother chicken',
 'age': 63,
 'is_able_to_fly': False,
 'created': datetime.datetime(2019, 3, 18, 11, 10, 31, 149995, tzinfo=datetime.timezone.utc),
 'updated': datetime.datetime(2019, 3, 18, 11, 10, 31, 149995, tzinfo=datetime.timezone.utc)}

Now we can retrieve the same document from database.

In [18]:
Bird.findOne({'name': 'mother chicken'})

{'_id': ObjectId('5c8f7ca7144682821ddf40d2'),
 'name': 'mother chicken',
 'age': 63,
 'is_able_to_fly': False,
 'created': datetime.datetime(2019, 3, 18, 11, 10, 31, 149000),
 'updated': datetime.datetime(2019, 3, 18, 11, 10, 31, 149000)}

It is the same chicken, isn't it? great. Let's clear (eat) it.

In [19]:
mother_chicken.remove()

1

In [20]:
if not Bird.findOne({'_id': mother_chicken._id}):
    print('Yes. The mother chicken not found. Someone might ate it.')

Yes. The mother chicken not found. Someone might ate it.


Now we get all chickens which we stored so far.

In [21]:
all_chickens = Bird.find({'name': 'chicken'}, sort=[('_id', 1)])

In [22]:
len(all_chickens)

1

Or we can count with count() method directly.

In [23]:
Bird.count({'name': 'chicken'})

1

Let's check if the latest chicken is equal to the one which we just saved.

In [24]:
all_chickens[-1]._id.generation_time == chicken._id.generation_time

True

Is that `True`, right?

#### Contextual database

MongoBase automatically creates mongodb client for each process.  
But in some cases, some instances must be written or read for a different client or db.  
If you use db context, it uses a designated database within the context.  
Let's get try on it.

In [25]:
with db_context(db_uri='localhost', db_name='test') as db:
    print(db)
    flamingo = Bird({'_id': ObjectId(), 'name': 'flamingo', 'age': 20})
    flamingo.save(db=db)
    
    flamingo.age = 23
    flamingo = flamingo.update(db=db)
    assert flamingo.age == 23, 'something wrong on update()'
    flamingo = Bird.findAndUpdateById(flamingo._id, {'age': 24}, db=db)
    assert flamingo.age == 24, 'something wrong on findAndUpdateById()'
    
    n_flamingo = Bird.count({'name': 'flamingo'}, db=db)
    print(f'{n_flamingo} flamingo found in the test database.')

n_flamingo = Bird.count({'name': 'flamingo'})
print(f'{n_flamingo} flamingo found in the default database.')
assert n_flamingo == 0

Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True, connecttimeoutms=10000, serverselectiontimeoutms=10000, sockettimeoutms=10000, socketkeepalive=True, maxidletimems=10000, maxpoolsize=200, minpoolsize=10, waitqueuemultiple=12, waitqueuetimeoutms=100), 'test')
44 flamingo found in the test database.
0 flamingo found in the default database.


#### Bulk Operation

Many insert operations takes a large computing cost. Fortunately, MongoDB provides an operation named "bulk write".  
It enables to insert many documents in one operation.

Bulk Insert

In [26]:
many_pigeon = []
for i in range(10000):
    many_pigeon += [Bird({'_id': ObjectId(), 'name': f'pigeon', 'age': i})]
print(many_pigeon[1])

{'_id': ObjectId('5c8f7ca7144682821ddf40d6'), 'name': 'pigeon', 'age': 1, 'is_able_to_fly': False, 'created': None, 'updated': None}


In [27]:
%%time
Bird.bulk_insert(many_pigeon)

CPU times: user 180 ms, sys: 16 ms, total: 196 ms
Wall time: 245 ms


10000

In [28]:
Bird.count({'name': 'pigeon'})

10000

Bulk Update

In [29]:
updates = []
for pigeon in many_pigeon:
    pigeon.age *= 3
    updates += [pigeon]

In [30]:
%%time
print(len(updates))
Bird.bulk_update(updates)

10000
UpdateOne({'_id': ObjectId('5c8f7ca7144682821ddf40d5')}, {'$set': {'name': 'pigeon', 'age': 0, 'is_able_to_fly': False, 'created': datetime.datetime(2019, 3, 18, 11, 10, 31, 462292, tzinfo=datetime.timezone.utc), 'updated': datetime.datetime(2019, 3, 18, 11, 10, 31, 797328, tzinfo=datetime.timezone.utc)}}, False, None, None)
CPU times: user 476 ms, sys: 16 ms, total: 492 ms
Wall time: 863 ms


Check if all ages are updated

In [31]:
%%time
for i, pigeon in enumerate(many_pigeon):
    check = Bird.findOne({'_id': pigeon._id})
    assert check.age == i*3

CPU times: user 3.87 s, sys: 244 ms, total: 4.12 s
Wall time: 5.11 s


No error? Cool.

In [32]:
Bird.delete({'name': 'pigeon'})

10000

#### Multi Threading and Processing

In [33]:
def breed(i):
    try:
        sparrow = Bird({'_id': ObjectId(), 'name': f'sparrow', 'age': 0})
        sparrow.save()
        sparrow.age += 1
        sparrow.update()
    except Exception as e:
        print(f'Exception occured. {e} in thread {threading.current_thread()}')
    else:
        print(f'{i} saved in thread {threading.current_thread()}.')

Threading (using the same memory space)

>The threading module uses threads, the multiprocessing module uses processes. The difference is that threads run in the same memory space, while processes have separate memory. This makes it a bit harder to share objects between processes with multiprocessing. Since threads use the same memory, precautions have to be taken or two threads will write to the same memory at the same time. This is what the global interpreter lock is for.

https://stackoverflow.com/questions/3044580/multiprocessing-vs-threading-python

In [34]:
%%time
for i in range(1000):
    t = threading.Thread(target=breed, name=f'breed sparrow {i}', args=(i,))
    t.start()
    
Bird.delete({'name':'sparrow'})

0 saved in thread <Thread(breed sparrow 0, started 140445443766016)>.
1 saved in thread <Thread(breed sparrow 1, started 140445203556096)>.
2 saved in thread <Thread(breed sparrow 2, started 140445195163392)>.
3 saved in thread <Thread(breed sparrow 3, started 140445186770688)>.
9 saved in thread <Thread(breed sparrow 9, started 140444793026304)>.
4 saved in thread <Thread(breed sparrow 4, started 140445178377984)>.
13 saved in thread <Thread(breed sparrow 13, started 140444281333504)>.
8 saved in thread <Thread(breed sparrow 8, started 140444801419008)>.
19 saved in thread <Thread(breed sparrow 19, started 140443752855296)>.
6 saved in thread <Thread(breed sparrow 6, started 140444818204416)>.
11 saved in thread <Thread(breed sparrow 11, started 140444776240896)>.
30 saved in thread <Thread(breed sparrow 30, started 140443182413568)>.
7 saved in thread <Thread(breed sparrow 7, started 140444809811712)>.
16 saved in thread <Thread(breed sparrow 16, started 140444256155392)>.
33 saved i

Multiprocessing (using the separated memory for each process)

>PyMongo is not fork-safe. Care must be taken when using instances of MongoClient with fork(). Specifically, instances of MongoClient must not be copied from a parent process to a child process. Instead, the parent process and each child process must create their own instances of MongoClient. Instances of MongoClient copied from the parent process have a high probability of deadlock in the child process due to the inherent incompatibilities between fork(), threads, and locks described below. PyMongo will attempt to issue a warning if there is a chance of this deadlock occurring.
http://api.mongodb.com/python/current/faq.html#pymongo-fork-safe%3E

In [35]:
def breed2(tasks):
    db = Bird._db()  # create a MongoDB Client for the forked process
    try:
        for i in range(len(tasks)):
            sparrow = Bird({'_id': ObjectId(), 'name': f'sparrow', 'age': 0})
            sparrow.save(db=db)
            sparrow.age += 1
            sparrow.update(db=db)
    except Exception as e:
        print(f'Exception occured. {e} in process {multiprocessing.current_process()}')
    else:
        print(f'{len(tasks)} sparrow saved in process {multiprocessing.current_process()}.')

12 saved in thread <Thread(breed sparrow 12, started 140444289726208)>.


In [36]:
%%time
print(f'{multiprocessing.cpu_count()} cpu resources found.')
tasks = [[f'sparrow {i}' for i in range(250)] for j in range(4)]
process_pool = multiprocessing.Pool(4)
process_pool.map(breed2, tasks)

40 cpu resources found.
48 saved in thread <Thread(breed sparrow 48, started 140441596978944)>.
52 saved in thread <Thread(breed sparrow 52, started 140441563408128)>.
34 saved in thread <Thread(breed sparrow 34, started 140442670720768)>.
35 saved in thread <Thread(breed sparrow 35, started 140442662328064)>.
26 saved in thread <Thread(breed sparrow 26, started 140443215984384)>.
38 saved in thread <Thread(breed sparrow 38, started 140442637149952)>.
27 saved in thread <Thread(breed sparrow 27, started 140443207591680)>.
100 saved in thread <Thread(breed sparrow 100, started 140437813704448)>.
62 saved in thread <Thread(breed sparrow 62, started 140440523237120)>.
40 saved in thread <Thread(breed sparrow 40, started 140442142242560)>.
66 saved in thread <Thread(breed sparrow 66, started 140440489666304)>.
20 saved in thread <Thread(breed sparrow 20, started 140443744462592)>.
10 saved in thread <Thread(breed sparrow 10, started 140444784633600)>.
44 saved in thread <Thread(breed sparr

250 sparrow saved in process <ForkProcess(ForkPoolWorker-1, started daemon)>.
Exception occured. Timed out waiting for socket from pool with max_size 200 and wait_queue_timeout 0.1 in thread <Thread(breed sparrow 884, started 140416621795072)>
Exception occured. Timed out waiting for socket from pool with max_size 200 and wait_queue_timeout 0.1 in thread <Thread(breed sparrow 29, started 140443190806272)>
250 sparrow saved in process <ForkProcess(ForkPoolWorker-3, started daemon)>.
CPU times: user 488 ms, sys: 412 ms, total: 900 ms
Wall time: 636 ms
250 sparrow saved in process <ForkProcess(ForkPoolWorker-2, started daemon)>.
250 sparrow saved in process <ForkProcess(ForkPoolWorker-4, started daemon)>.
101 saved in thread <Thread(breed sparrow 101, started 140437805311744)>.
